**This pipeline was ultimately abandoned due to severe preprocessing latency and unresolved memory limits (OOM errors) during the transformation phase. This led to the architectural decision to pivot toward more computationally efficient models.**

In [ ]:
import time
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import random
import joblib
from pathlib import Path
from torchvision.models import resnet18

import matplotlib.pyplot as plt
import glob

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.metrics import cohen_kappa_score,f1_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler,MinMaxScaler

import tensorflow as tf
from keras.callbacks import Callback
device = torch.device("cuda")


import numpy as np

def generate_gaf(time_series, method='summation'):
    """
    Encodes a 1D time series into a 2D Gramian Angular Field matrix.
    
    Parameters:
    time_series (array-like): 1D array of time series values.
    method (str): 'summation' (GASF) or 'difference' (GADF).
    
    Returns:
    np.ndarray: 2D GAF matrix of shape (N, N).
    """
    # 1. Normalize the time series to the interval [-1, 1]
    ts_min, ts_max = np.min(time_series), np.max(time_series)
    ts_norm = 2 * (time_series - ts_min) / (ts_max - ts_min) - 1
    
    # Clip to avoid floating point issues slightly outside [-1, 1]
    ts_norm = np.clip(ts_norm, -1, 1)
    
    # 2. Convert to polar coordinates (angles)
    angles = np.arccos(ts_norm)
    
    # 3. Create the Gramian Angular Field
    n = len(angles)
    
    if method == 'summation':
        # GASF: cos(theta_i + theta_j)
        angles_matrix = angles[:, np.newaxis] + angles[np.newaxis, :]
        gaf_matrix = np.cos(angles_matrix)
    elif method == 'difference':
        # GADF: sin(theta_i - theta_j)
        angles_matrix = angles[:, np.newaxis] - angles[np.newaxis, :]
        gaf_matrix = np.sin(angles_matrix)
    else:
        raise ValueError("Method must be either 'summation' or 'difference'")
        
    return gaf_matrix



In [ ]:

# REDUCED_DATA = 250
COMPETITION_DATA_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction"
TRAIN_DIR            = os.path.join(COMPETITION_DATA_DIR, "train")
TEST_DIR             = os.path.join(COMPETITION_DATA_DIR, "test")
SAMPLE_SUBMISSION    = os.path.join(COMPETITION_DATA_DIR, "sample_submission.csv")
SUBMISSION_FILE      = "submission.csv"

WINDOW_SIZE = 64
EPOCHS = 20
BATCH_SIZE = 16
#OPTIMIZER
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5


wellname = set()

for file in os.listdir(TRAIN_DIR):
    if file[-3:] != "png":
        wellname.add(file.split("__")[0])
wellname = list(wellname)
wellname = wellname[:int(len(wellname)*0.5)]

In [ ]:
# from torch.utils.data import Dataset, DataLoader

# class GeosteeringImageDataset(Dataset):
#     def __init__(self, manifest_path, typewell_dictionary):
#         # Load the CSV map
#         self.manifest = pd.read_csv(manifest_path)
#         self.typewell_dict = typewell_dictionary
        
#     def __len__(self):
#         return len(self.manifest)
        
#     def __getitem__(self, idx):
#         # 1. Read the row from the CSV
#         row = self.manifest.iloc[idx]
        
#         # 2. Load the 6-channel image from the hard drive
#         image_np = np.load(row['filepath'])
#         image_tensor = torch.tensor(image_np, dtype=torch.float32)
        
#         # 3. Look up the matching Typewell using the well name
#         well_name = row['well_name']
#         typewell_np = self.typewell_dict[well_name]
#         typewell_tensor = torch.tensor(np.transpose(typewell_np, (1,0)), dtype=torch.float32)
        
#         # 4. Grab the target
#         target_drift = torch.tensor([row['target_drift']], dtype=torch.float32)
        
#         target_forms = torch.tensor([
#             row['target_form_0'], row['target_form_1'], 
#             row['target_form_2'], row['target_form_3'], 
#             row['target_form_4']
#         ], dtype=torch.float32)
        
#         return image, typewell, target_drift, target_forms


In [ ]:
# Since it's a Dataset, we just point directly to the exact file path you copied
RESNET18_WEIGHTS = Path("/kaggle/input/datasets/organizations/pytorch/resnet18/resnet18.pth")

print("Using ResNet weights at:", RESNET18_WEIGHTS)
print("Exists:", RESNET18_WEIGHTS.exists())
HAS_PRETRAINED = RESNET18_WEIGHTS.is_file()
print("HAS_PRETRAINED", HAS_PRETRAINED)

class ResNetExtractor(nn.Module):
    def __init__(self, weights_path=None, in_channels=6):
        super().__init__()
        
        # Initialize the raw architecture
        self.backbone = resnet18(weights=None)
        
        # Load the offline Kaggle weights if provided
        if weights_path is not None:
            print(f"Loading ResNet18 backbone from {weights_path}")
            state = torch.load(weights_path, map_location="cpu", weights_only=False)
            # Drop the fully connected layer from the state dict
            state = {k: v for k, v in state.items() if not k.startswith("fc.")}
            self.backbone.load_state_dict(state, strict=False)
            print("Loaded ResNet backbone.")
        
        # Modify the first layer to accept our 6 GAF channels
        self.backbone.conv1 = nn.Conv2d(
            in_channels=in_channels, 
            out_channels=64, 
            kernel_size=7, 
            stride=2, 
            padding=3, 
            bias=False
        )
        
        # Strip the classification head so it outputs the raw 512 feature vector
        self.backbone.fc = nn.Identity()

    def forward(self, x):
        # Simply pass the image through the modified backbone
        return self.backbone(x)

class SimpleConv1d(nn.Module):
    def __init__(self, in_channels=2): # 2 channels: GR and TVT
        super().__init__()
        
        self.feature_extractor = nn.Sequential(
            nn.Conv1d(in_channels, out_channels=16, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1) 
        )
        
        self.flatten = nn.Flatten()
        
        # Map the 32 channels to 512 to perfectly match the ResNet18 output
        self.fc = nn.Linear(in_features=32, out_features=512)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.feature_extractor(x)
        x = self.flatten(x)
        x = self.fc(x)
        x = self.relu(x)
        return x

class SiameseGeosteeringNet(nn.Module): # FIX: Added nn.Module
    def __init__(self, weights_path=None):
        super().__init__()
 
        self.branch1 = ResNetExtractor(weights_path=weights_path, in_channels=6)
        self.branch2 = SimpleConv1d(in_channels=2) 
   
        self.merged_layer = nn.Sequential(
            nn.Linear(in_features=1024, out_features=256), # 512 (Image) + 512 (Typewell)
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        self.drift_head = nn.Linear(in_features=256, out_features=1)
        self.form_head = nn.Linear(in_features=256, out_features=5)
       
    def forward(self, image, typewell, inference_only=False):
        
        img_features = self.branch1(image)         # Shape: (batch, 512)
        type_features = self.branch2(typewell)     # Shape: (batch, 512)

        combined = torch.cat((img_features, type_features), dim=1) # Shape: (batch, 1024)
        merged_out = self.merged_layer(combined)   # Shape: (batch, 256)

        pred_drift = self.drift_head(merged_out)   # Shape: (batch, 1)
        
        if inference_only:
            return pred_drift
            
        pred_forms = self.form_head(merged_out)    # Shape: (batch, 5)
        return pred_drift, pred_forms

In [ ]:
def make_sliding_windows(df, feature_cols, target_tvt_col, target_form_col, window_size=64, step=16):
    x_windows, y_tvt_drift_windows, y_form_windows, anchor_vals = [], [], [], []
    
    for i in range(0, len(df) - window_size + 1, step):
        window = df.iloc[i : i + window_size].copy()
        if random.random() > 0.5:
            blind_start = random.randint(10, window_size - 10)
            anchor_val = window['TVT_input'].iloc[blind_start - 1]
        else:
            anchor_val = window['TVT_input'].iloc[-1]

        final_tvt = window[target_tvt_col].values[-1]
        drift_target = final_tvt - anchor_val

        x_windows.append(window[feature_cols].values)
        y_tvt_drift_windows.append(drift_target) 
        y_form_windows.append(window[target_form_col].values[-1])     
        anchor_vals.append(anchor_val)

    return np.array(x_windows), np.array(y_tvt_drift_windows), np.array(y_form_windows), np.array(anchor_vals)

features = ['GR', 'Z', 'MD', 'TVT_input'] 
formations = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']
new_formation_targets = [f'dist_{form}' for form in formations]


max_type_len = 0
for well in wellname:
    t_len = len(pd.read_csv(f"{TRAIN_DIR}/{well}__typewell.csv"))
    if t_len > max_type_len:
        max_type_len = t_len
print(f"Maximum Typewell Length: {max_type_len}")

In [ ]:
well_data_cache = {}
all_train_drifts = []
all_train_features = []
all_train_forms = []
typewell_dict = {}

for well in wellname:
    horiz_df = pd.read_csv(f"{TRAIN_DIR}/{well}__horizontal_well.csv")
    type_df = pd.read_csv(f"{TRAIN_DIR}/{well}__typewell.csv")
    
    cols_to_clean = features + formations + ['TVT']
    for col in cols_to_clean:
        horiz_df[col] = horiz_df[col].interpolate().ffill().bfill().fillna(0)
    type_df['GR'] = type_df['GR'].interpolate().ffill().bfill().fillna(0)
    type_df['TVT'] = type_df['TVT'].interpolate().ffill().bfill().fillna(0)

    for form in formations:
        horiz_df[f'dist_{form}'] = horiz_df[form] - horiz_df['Z']

    x_horiz, y_drift, y_form_dists, anchors = make_sliding_windows(
        horiz_df, feature_cols=features, target_tvt_col='TVT', 
        target_form_col=new_formation_targets, window_size=WINDOW_SIZE, step=16
    )
    
    well_data_cache[well] = {
        'x_horiz': x_horiz,
        'y_drift': y_drift,
        'y_form_dists': y_form_dists,
        'anchors': anchors
    }

    all_train_drifts.append(y_drift)
    all_train_forms.append(y_form_dists)
    all_train_features.append(horiz_df[features].values)

    type_array = type_df[['GR', 'TVT']].values
    actual_len = len(type_array)
    if actual_len < max_type_len:
        pad_size = max_type_len - actual_len
        pad_values = np.tile(type_array[-1], (pad_size, 1))
        type_array = np.vstack([type_array, pad_values])

    typewell_dict[well] = type_array
    
# Combine only the training targets
train_drifts_combined = np.concatenate(all_train_drifts, axis=0)
train_forms_combined = np.concatenate(all_train_forms, axis=0)
train_features_combined = np.concatenate(all_train_features,axis = 0)


scaler_horiz = MinMaxScaler(feature_range=(-1, 1))
scaler_horiz.fit (train_features_combined)
# Globally fit the scalers!
scaler_target_drift = StandardScaler()
scaler_target_drift.fit(train_drifts_combined.reshape(-1, 1))

scaler_target_form = StandardScaler()
scaler_target_form.fit(train_forms_combined) 

# Save the drift scaler for your inference notebook
joblib.dump(scaler_horiz,'/kaggle/working/scaler_horiz.pkl')
joblib.dump(scaler_target_drift, '/kaggle/working/scaler_target_drift.pkl')

In [ ]:
class GeosteeringImageDataset(Dataset):
    def __init__(self, manifest_df, raw_windows_list, typewell_dictionary, gasf, gadf):
        self.manifest = manifest_df
        self.raw_windows = raw_windows_list
        self.typewell_dict = typewell_dictionary
        self.gasf = gasf
        self.gadf = gadf

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        row = self.manifest[idx]
        
        # 1. Grab the tiny 64x4 raw window
        raw_window = self.raw_windows[idx]
        
        # 2. GENERATE THE IMAGE ON THE FLY!
        image_np = generate_single_gaf(self.gasf, self.gadf, raw_window)
        image_tensor = torch.tensor(image_np, dtype=torch.float32)
             
        # 3. Look up the matching Typewell using the well name
        well_name = row['well_name']
        typewell_np = self.typewell_dict[well_name]
        typewell_tensor = torch.tensor(np.transpose(typewell_np, (1,0)), dtype=torch.float32)
        
        # 4. Grab the target
        target_drift = torch.tensor([row['target_drift']], dtype=torch.float32)
        
        target_forms = torch.tensor([
            row['target_form_0'], row['target_form_1'], 
            row['target_form_2'], row['target_form_3'], 
            row['target_form_4']
        ], dtype=torch.float32)
  
        
        return image_tensor, typewell_tensor, target_drift, target_forms


In [ ]:

# def generate_single_gaf(gasf, gadf, single_window):
#     # single_window shape: (64, 4)
#     # PyTS expects 2D inputs of shape (n_samples, n_timestamps), so we reshape to (1, 64)
#     gr_feat = single_window[:, 0].reshape(1, -1)
#     z_feat  = single_window[:, 1].reshape(1, -1)
#     md_feat = single_window[:, 2].reshape(1, -1)
    
#     # Generate the fields and grab the first (and only) item in the batch
#     gr_gasf = gasf.fit_transform(gr_feat)[0]
#     gr_gadf = gadf.fit_transform(gr_feat)[0]
#     z_gasf  = gasf.fit_transform(z_feat)[0]
#     z_gadf  = gadf.fit_transform(z_feat)[0]
#     md_gasf = gasf.fit_transform(md_feat)[0]
#     md_gadf = gadf.fit_transform(md_feat)[0]
    
#     # Stack along the channel axis (shape becomes 6, 64, 64)
#     return np.stack([gr_gasf, gr_gadf, z_gasf, z_gadf, md_gasf, md_gadf], axis=0)

train_wells, val_wells = train_test_split(wellname, train_size=0.8, random_state=42)
train_well_set = set(train_wells)

# train_raw_windows = []
# val_raw_windows = []
# train_manifest = []
# val_manifest = []
# Example usage with a dummy time series

# gasf = GramianAngularField(image_size=64, method='summation', sample_range=None)
# gadf = GramianAngularField(image_size=64, method='difference', sample_range=None)


# for well, data in well_data_cache.items():
#     # 1. Transform the targets using the globally fitted scalers
#     y_drift_scaled = scaler_target_drift.transform(data["y_drift"].reshape(-1, 1)).flatten()
#     y_form_scaled = scaler_target_form.transform(data["y_form_dists"])

#     raw_x = data['x_horiz'] 
#     N = raw_x.shape[0]
    
#     # Flatten -> Scale -> Reshape back to 3D
#     flat_x = raw_x.reshape(-1, 4) 
#     scaled_flat_x = scaler_horiz.transform(flat_x)
#     scaled_x_horiz = scaled_flat_x.reshape(N, 64, 4)
    
#     num_windows = scaled_x_horiz.shape[0]
    
#     for i in range(num_windows):
#         record = {
#             'well_name': well,
#             'target_drift': y_drift_scaled[i],
#             'target_form_0': y_form_scaled[i][0], 
#             'target_form_1': y_form_scaled[i][1],
#             'target_form_2': y_form_scaled[i][2],
#             'target_form_3': y_form_scaled[i][3],
#             'target_form_4': y_form_scaled[i][4],
#             'anchor_value': data['anchors'][i]    

#         }
        
#         # Save ONLY the raw 64x4 array (takes almost no memory!)
#         if well in train_well_set:
#             train_manifest.append(record)
#             train_raw_windows.append(scaled_x_horiz[i]) 
#         else:
#             val_manifest.append(record)
#             val_raw_windows.append(scaled_x_horiz[i])
            
# pd.DataFrame(train_manifest).to_csv("train_manifest.csv", index=False)
# pd.DataFrame(val_manifest).to_csv("val_manifest.csv", index=False)

In [ ]:
def generate_gaf_image (gasf,gadf,x_horiz):
    # 2. Generate the massive images
    gr_feat = x_horiz[:, :, 0]
    z_feat = x_horiz[:, :, 1]
    md_feat = x_horiz[:, :, 2]

    print(f"GAF Matrix shape: {gaf_sum.shape}")

    gr_gasf = generate_gaf(gr_feat, method='summation')
    gr_gadf = generate_gaf(gr_feat, method='difference')

    z_gasf = generate_gaf(z_feat, method='summation')
    z_gadf = generate_gaf(z_feat, method='difference')

    md_gasf = generate_gaf(md_feat, method='summation')
    md_gadf = generate_gaf(md_feat, method='difference')
    
    
    # gr_gasf = gasf.fit_transform(gr_feat)
    # gr_gadf = gadf.fit_transform(gr_feat)
    
    # z_gasf = gasf.fit_transform(z_feat)
    # z_gadf = gadf.fit_transform(z_feat)
    
    # md_gasf = gasf.fit_transform(md_feat)
    # md_gadf = gadf.fit_transform(md_feat)
    
    stacked_images = np.stack([gr_gasf, gr_gadf, z_gasf, z_gadf, md_gasf, md_gadf], axis=1)
    return stacked_images
    

# train_manifest = []
# val_manifest = []

# train_wells, val_wells = train_test_split(wellname, train_size=0.8, random_state=42)
# train_well_set = set(train_wells)

# # Initialize pyts
# gasf = GramianAngularField(image_size=64, method='summation', sample_range=None)
# gadf = GramianAngularField(image_size=64, method='difference', sample_range=None)

# os.makedirs("image_dataset", exist_ok=True)
# for well, data in well_data_cache.items():
#     # 1. Transform the targets using the globally fitted scalers
#     y_drift_scaled = scaler_target_drift.transform(data["y_drift"].reshape(-1, 1)).flatten()
#     y_form_scaled = scaler_target_form.transform(data["y_form_dists"])

#     raw_x = data['x_horiz'] 
#     N = raw_x.shape[0]
    
#     # Flatten -> Scale -> Reshape back to 3D
#     flat_x = raw_x.reshape(-1, 4) 
#     scaled_flat_x = scaler_horiz.transform(flat_x)
#     scaled_x_horiz = scaled_flat_x.reshape(N, 64, 4)
    
#     stacked_images = generate_gaf_image (gasf,gadf,scaled_x_horiz)
    
#     # 3. Save to disk and append to the correct manifest
#     num_windows = stacked_images.shape[0]
#     for i in range(num_windows):
#         filename = f"{well}_window_{i}.npy"
#         filepath = f"image_dataset/{filename}"
        
#         np.save(filepath, stacked_images[i])
        
#         # FIX: Added all 5 formations
#         record = {
#             'filepath': filepath,
#             'well_name': well,
#             'target_drift': y_drift_scaled[i], 
#             'target_form_0': y_form_scaled[i][0], 
#             'target_form_1': y_form_scaled[i][1],
#             'target_form_2': y_form_scaled[i][2],
#             'target_form_3': y_form_scaled[i][3],
#             'target_form_4': y_form_scaled[i][4],
#             'anchor_value': data['anchors'][i]    
#         }
        
#         if well in train_well_set:
#             train_manifest.append(record)
#         else:
#             val_manifest.append(record)

# # Save the CSVs
# pd.DataFrame(train_manifest).to_csv("train_manifest.csv", index=False)
# pd.DataFrame(val_manifest).to_csv("val_manifest.csv", index=False)

In [ ]:
class GeosteeringImageDataset(Dataset):
    def __init__(self, manifest_list, typewell_dictionary, h5_path="geosteering_cache.h5"):
        self.manifest = manifest_list
        self.typewell_dict = typewell_dictionary
        self.h5_path = h5_path
        self.h5_file = None # We open this lazily below!

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        # Open the file only once per worker thread
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
            
        row = self.manifest[idx]
        well_name = row['well_name']
        window_idx = row['window_index']
        
--
        # HDF5 reaches into the disk and pulls ONLY this single 6x64x64 array into RAM
        image_np = self.h5_file[well_name][window_idx] 
        image_tensor = torch.tensor(image_np, dtype=torch.float32)
        
        # Look up matching Typewell
        typewell_np = self.typewell_dict[well_name]
        typewell_tensor = torch.tensor(np.transpose(typewell_np, (1, 0)), dtype=torch.float32)
        
        # Grab targets
        target_drift = torch.tensor([row['target_drift']], dtype=torch.float32)
        target_forms = torch.tensor([
            row['target_form_0'], row['target_form_1'], 
            row['target_form_2'], row['target_form_3'], row['target_form_4']
        ], dtype=torch.float32)
        
        return image_tensor, typewell_tensor, target_drift, target_forms

In [ ]:
import h5py
import os

# Delete old h5 file if you rerun the cell so it doesn't duplicate
if os.path.exists("geosteering_cache.h5"):
    os.remove("geosteering_cache.h5")

train_manifest = []
val_manifest = []
# gasf = GramianAngularField(image_size=64, method='summation', sample_range=(-1, 1))
# gadf = GramianAngularField(image_size=64, method='difference', sample_range=(-1, 1))

# Open the master HDF5 file in write mode
counter = 1
with h5py.File("geosteering_cache.h5", "w") as hf:
    
    for well, data in well_data_cache.items():
        print (counter)
        counter+=1
        # 1. Transform targets
        y_drift_scaled = scaler_target_drift.transform(data["y_drift"].reshape(-1, 1)).flatten()
        y_form_scaled = scaler_target_form.transform(data["y_form_dists"])
        
        # 2. Extract and scale features
        raw_x = data['x_horiz']
        N = raw_x.shape[0]
        flat_x = raw_x.reshape(-1, 4)
        scaled_flat_x = scaler_horiz.transform(flat_x)
        scaled_flat_x = np.clip(scaled_flat_x, -1.0, 1.0)
        scaled_x_horiz = scaled_flat_x.reshape(N, 64, 4)
        
        # 3. Generate ALL images for this well at once
        # stacked_images = generate_gaf_image(gasf, gadf, scaled_x_horiz)
        stacked_images =generate_gaf_image (scaled_x_horiz)
        
        # 4. Save to the HDF5 file! 
        # compression="lzf" makes the file tiny on disk but very fast to read
        hf.create_dataset(well, data=stacked_images.astype(np.float32), compression="lzf")
        
        # 5. Build Manifests
        for i in range(N):
            record = {
                'well_name': well,
                'window_index': i, # <--- WE NEED THIS TO FIND IT IN THE H5 FILE LATER
                'target_drift': y_drift_scaled[i],
                'target_form_0': y_form_scaled[i][0],
                'target_form_1': y_form_scaled[i][1],
                'target_form_2': y_form_scaled[i][2],
                'target_form_3': y_form_scaled[i][3],
                'target_form_4': y_form_scaled[i][4],
            }
            if well in train_well_set:
                train_manifest.append(record)
            else:
                val_manifest.append(record)

print("HDF5 Cache built successfully!")

In [ ]:
# # train_dataset = GeosteeringImageDataset (
# #     manifest_path = "train_manifest.csv",
# #     typewell_dictionary = typewell_dict,
# # )

# train_dataset = GeosteeringImageDataset(
#     train_manifest, train_raw_windows, typewell_dict, 
#     gasf, gadf)

# # val_dataset = GeosteeringImageDataset (
# #     manifest_path = "val_manifest.csv" ,
# #     typewell_dictionary = typewell_dict,
# # )
# val_dataset = GeosteeringImageDataset(
#     val_manifest, val_raw_windows, typewell_dict, 
#     gasf, gadf)


# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)

train_dataset = GeosteeringImageDataset(train_manifest, typewell_dict)
val_dataset = GeosteeringImageDataset(val_manifest, typewell_dict)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
model = SiameseGeosteeringNet(weights_path = RESNET18_WEIGHTS)
model = model.to(device) 

optimizer = torch.optim.Adam(model.parameters(),lr=LEARNING_RATE, weight_decay = WEIGHT_DECAY)
scheduler = ReduceLROnPlateau(optimizer, 'min', patience=10, factor=0.8, min_lr=1e-8)
    
criterion_drift = nn.MSELoss()
criterion_form = nn.MSELoss()
early_stopping_patience = 20
best_val_loss = float('inf')
epochs_no_improve = 0
for epoch in range(EPOCHS):
    start_time = time.time()
    model.train()
    train_loss = 0.0
    for x_image,x_typewell, y_drift, y_forms in train_loader:
        x_image, x_typewell = x_image.to(device), x_typewell.to(device)
        y_drift, y_forms = y_drift.to(device), y_forms.to(device)

        pred_drift, pred_forms = model (x_image,x_typewell, inference_only = False)

        loss_main = criterion_drift(pred_drift,y_drift)
        loss_aux = criterion_form(pred_forms, y_forms)

        total_loss = loss_main + (loss_aux * 0.2)

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        train_loss += total_loss.item()/len(train_loader)

    val_loss = 0.0
    val_tvt_rmse = 0.0
    model.eval()
    with torch.no_grad():
        for x_image,x_typewell, y_drift, y_forms in val_loader:
            x_image, x_typewell = x_image.to(device), x_typewell.to(device)
            y_drift, y_forms = y_drift.to(device), y_forms.to(device)
    
            pred_drift, pred_forms = model (x_image,x_typewell, inference_only = False)

            loss_main = criterion_drift(pred_drift,y_drift)
            loss_aux = criterion_form(pred_forms, y_forms)
    
            total_loss = loss_main + (loss_aux * 0.2)
            val_loss += total_loss.item()/len(val_loader)
            
            pred_drift_unscaled = pred_drift.cpu().numpy() * scaler_target_drift.scale_[0] + scaler_target_drift.mean_[0]
            y_drift_unscaled = y_drift.cpu().numpy() * scaler_target_drift.scale_[0] + scaler_target_drift.mean_[0]
            batch_rmse = np.sqrt(np.mean((pred_drift_unscaled - y_drift_unscaled)**2))
            
            val_tvt_rmse += batch_rmse / len(val_loader)

    scheduler.step(val_loss)
    elapsed = time.time() - start_time
        
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_geosteering_model.pt')

        with open('best_model_stats.txt', 'w') as log_file:
            log_file.write(f"Best Epoch: {epoch + 1}\n")
            log_file.write(f"Train Loss: {train_loss:.4f}\n")
            log_file.write(f"Val Loss: {val_loss:.4f}\n")
            log_file.write(f"Val TVT RMSE: {val_tvt_rmse:.2f} ft\n")
    else:
        epochs_no_improve += 1
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val TVT RMSE: {val_tvt_rmse:.2f} ft | Time: {elapsed:.1f}s")
    if epochs_no_improve >= early_stopping_patience:
        print(f"\nEarly stopping triggered! No improvement in validation loss for {early_stopping_patience} epochs.")
        break        

In [ ]:
import gc
from collections import defaultdict



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
max_type_len = max(len(pd.read_csv(f"{TRAIN_DIR}/{w}__typewell.csv")) for w in wellname)

model = SiameseGeosteeringNet(weights_path = RESNET18_WEIGHTS)
model = model.to(device)

if os.path.exists('best_geosteering_model.pt'):
    print(f"Loading saved model...")
    model.load_state_dict(torch.load('best_geosteering_model.pt', map_location=device))

model.eval()

sample_sub = pd.read_csv(SAMPLE_SUBMISSION)
well_rows = defaultdict(list)
split = sample_sub['id'].str.rsplit('_', n=1, expand=True)
for well, row_num in zip(split[0], split[1]):
    well_rows[well].append(int(row_num))

pd.DataFrame(columns=['id', 'tvt']).to_csv("submission.csv", index=False)

gasf = GramianAngularField(image_size=64, method='summation', sample_range=None)
gadf = GramianAngularField(image_size=64, method='difference', sample_range=None)
with torch.no_grad():
    for well in well_rows.keys():
        horiz_df = pd.read_csv(f"{TEST_DIR}/{well}__horizontal_well.csv")
        type_df = pd.read_csv(f"{TEST_DIR}/{well}__typewell.csv")
        

        for col in features:
            horiz_df[col] = horiz_df[col].interpolate().ffill().bfill().fillna(0)
        type_df['GR'] = type_df['GR'].interpolate().ffill().bfill().fillna(0)
        type_df['TVT'] = type_df['TVT'].interpolate().ffill().bfill().fillna(0)

        raw_tvt_input = horiz_df['TVT_input'].values.copy()
        
        horiz_df[features] = scaler_horiz.transform(horiz_df[features].values)

        # PREPARE THE TYPEWELL TENSOR
        type_array = type_df[['GR', 'TVT']].values
        actual_type_len = len(type_array)
        if actual_type_len < max_type_len:
            pad_size = max_type_len - actual_type_len
            type_array = np.vstack([type_array, np.tile(type_array[-1], (pad_size, 1))])
        
        # Shape becomes (1, 2, 10000)
        x_type = torch.tensor(type_array.T, dtype=torch.float32).unsqueeze(0).to(device)

        # CHUNK & PREDICT
        num_rows = len(horiz_df)
        predicted_tvt_full = np.zeros(num_rows)
        
        for i in range(WINDOW_SIZE, num_rows):
            
            # Extract the 64-foot window ending at the current row `i`
            window = horiz_df[features].iloc[i - WINDOW_SIZE : i].values
            anchor_val = raw_tvt_input[i - 1] 

            # stacked_np = generate_gaf_image(gasf,gadf,window)
            stacked_np = generate_gaf_image(window)
            # Convert to PyTorch Tensor AFTER image generation: shape (1, 6, 64, 64)
            stacked_images = torch.tensor(stacked_np, dtype=torch.float32).unsqueeze(0).to(device)
            
            pred_drift_tensor = model(stacked_images, x_type, inference_only=True)
            pred_drift_scaled = pred_drift_tensor.cpu().numpy()
            pred_drift_ft = scaler_target_drift.inverse_transform(pred_drift_scaled).flatten()[0]

            predicted_tvt_full[i] = pred_drift_ft + anchor_val

        well_df = pd.DataFrame({
            "id":  [f"{well}_{r}" for r in well_rows[well]],
            "tvt": [predicted_tvt_full[r] for r in well_rows[well]]
        })
        
        well_df.to_csv("submission.csv", mode='a', header=False, index=False)
        

        del horiz_df, type_df, predicted_tvt_full, well_df, x_type, stacked_images
        gc.collect()